# nerf_data — Getting Started

This notebook gives an interactive tour of the `nerf_data` toolkit for X-ray NeRF data preparation.

**What you'll learn:**
1. Loading and inspecting `transforms.json` (the NeRF camera-pose format)
2. Plotting camera positions and trajectories in 3D
3. Parsing an object-collection YAML (scene geometry definition)
4. Understanding the raw ↔ npy axis-order convention
5. How `compute_transforms.py` builds a camera matrix from X-ray hardware metadata
6. How `combine_transforms.py` merges per-timestep files

All cells run on the data already in this repository — no external downloads needed.

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/igrega348/nerf_data/HEAD?labpath=examples%2Fgetting_started.ipynb)

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml
from scipy.spatial.transform import Rotation

# Resolve repo root: works whether notebook is run from examples/ or repo root
HERE = Path('.').resolve()
REPO = HERE.parent if (HERE / '..').joinpath('synthetic').exists() else HERE
if not (REPO / 'synthetic').exists():
    REPO = HERE  # fallback: assume cwd is repo root
print('Repo root:', REPO)

---
## 1. Loading `transforms.json`

Every dataset is described by a `transforms.json` file in the [nerfstudio](https://github.com/nerfstudio-project/nerfstudio) format, extended with an optional `time` field for 4D (time-resolved) datasets.

We'll use the **synthetic balls** dataset — six spheres with different radii, 281 projections.

In [ ]:
transforms_path = REPO / 'synthetic' / 'balls' / 'transforms.json'
with open(transforms_path) as f:
    transforms = json.load(f)

print('Top-level keys:', list(transforms.keys()))
print(f"Image size:     {transforms['w']} × {transforms['h']} px")
print(f"Focal length:   fl_x={transforms['fl_x']:.2f}  fl_y={transforms['fl_y']:.2f}  (px)")
print(f"FoV (horiz):    {math.degrees(transforms['camera_angle_x']):.1f}°")
print(f"Frames:         {len(transforms['frames'])}")

In [ ]:
# Inspect the first frame
frame0 = transforms['frames'][0]
print('First frame keys:', list(frame0.keys()))
print('File path:', frame0['file_path'])
print('Transform matrix (4×4):')
M = np.array(frame0['transform_matrix'])
print(np.array2string(M, precision=4, suppress_small=True))
print('\nCamera origin (M[:3, 3]):', M[:3, 3])

### Camera-to-world matrix layout

```
M = [ R  |  t ]
    [ 0  |  1 ]
```

- `M[:3, :3]` — rotation (camera-to-world)
- `M[:3, 3]` — camera origin in world coordinates
- Camera looks along its local **+Z** axis

The world origin is at the centre of the scanned object; the camera orbits at radius `R`.

In [ ]:
# Extract all camera origins
origins = np.array([f['transform_matrix'] for f in transforms['frames']])[:, :3, 3]
print(f'Camera positions shape: {origins.shape}')
radii = np.linalg.norm(origins, axis=1)
print(f'Radius from origin:     min={radii.min():.3f}  max={radii.max():.3f}  mean={radii.mean():.3f}')

# Count train vs eval frames
train = [f for f in transforms['frames'] if 'train' in Path(f['file_path']).stem]
eval_ = [f for f in transforms['frames'] if 'eval'  in Path(f['file_path']).stem]
print(f'Train frames: {len(train)}   Eval frames: {len(eval_)}')

---
## 2. Visualising camera positions in 3D

For a circular CT scan the cameras trace a ring in the equatorial plane (`z ≈ 0`).

In [ ]:
fig = plt.figure(figsize=(10, 4))

# 3D scatter
ax3d = fig.add_subplot(121, projection='3d')
ax3d.scatter(origins[:, 0], origins[:, 1], origins[:, 2],
             s=6, c=np.arange(len(origins)), cmap='hsv', alpha=0.7)
ax3d.scatter(0, 0, 0, s=80, c='red', marker='*', label='object centre')
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')
ax3d.set_title('Camera positions (3D)')
ax3d.legend()

# Top-down (XY) view
ax2d = fig.add_subplot(122)
ax2d.scatter(origins[:, 0], origins[:, 1],
             s=6, c=np.arange(len(origins)), cmap='hsv', alpha=0.7)
ax2d.scatter(0, 0, s=80, c='red', marker='*')
ax2d.set_aspect('equal')
ax2d.set_xlabel('X'); ax2d.set_ylabel('Y')
ax2d.set_title('Top-down view (XY plane)')

plt.tight_layout()
plt.show()

Each colour step represents one projection. The cameras lie in the equatorial plane and span a full 360° rotation.

---
## 3. Parsing a scene YAML

The scene geometry is described in a YAML file as an `object_collection` of primitives. This is used both for forward rendering (X-ray simulation) and for volumetric supervision during NeRF training.

In [ ]:
yaml_path = REPO / 'synthetic' / 'balls' / 'balls.yaml'
with open(yaml_path) as f:
    scene = yaml.safe_load(f)

print('Scene type:', scene['type'])
print(f"Objects ({len(scene['objects'])}):\n")
for i, obj in enumerate(scene['objects']):
    print(f"  [{i}] type={obj['type']:10s}  radius={obj['radius']:.2f}  "
          f"center=[{obj['center'][0]:+.3f}, {obj['center'][1]:+.3f}, {obj['center'][2]:+.3f}]  "
          f"rho={obj['rho']}")

In [ ]:
# Visualise sphere positions and sizes
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection='3d')

for obj in scene['objects']:
    cx, cy, cz = obj['center']
    r = obj['radius']
    ax.scatter(cx, cy, cz, s=(r * 500) ** 2, alpha=0.5)

ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Scene objects (bubble size ∝ radius)')
plt.tight_layout()
plt.show()

The `rho` field is the linear X-ray attenuation coefficient (Beer-Lambert law). Higher `rho` = more attenuating = darker in the projection image.

---
## 4. Raw ↔ npy axis-order convention

X-ray CT scanners write volumetric data as flat binary (`.raw`) in `(z, y, x)` row-major order (z varies slowest). NumPy conventionally uses `(x, y, z)`. The scripts handle the swap automatically, but it's worth understanding:

| Format | Shape | Axis order |
|---|---|---|
| `.raw` from scanner | flat uint8/16 | `(Nz, Ny, Nx)` = z-major |
| `.npy` / `.npz` (numpy) | `(Nx, Ny, Nz)` | x-major |
| `.raw` written for renderer | flat uint8 | `(Nz, Ny, Nx)` = z-major |

In [ ]:
# Demonstrate the axis swap with a tiny asymmetric example
Nx, Ny, Nz = 3, 4, 5

# Scanner writes Nz*Ny*Nx bytes in (z, y, x) order
raw_flat = np.arange(Nx * Ny * Nz, dtype=np.uint8)

# raw_to_npy.py does:
#   vol = data.reshape([resolution[i] for i in [2,1,0]])  # → (Nz, Ny, Nx)
#   vol = vol.swapaxes(0, 2)                               # → (Nx, Ny, Nz)
vol_raw_order = raw_flat.reshape(Nz, Ny, Nx)   # as written by scanner
vol_npy = vol_raw_order.swapaxes(0, 2)          # → (Nx, Ny, Nz), stored in .npz as 'vol'

print(f'After reshape (z,y,x):  {vol_raw_order.shape}')
print(f'After swapaxes (x,y,z): {vol_npy.shape}  ← stored in .npz as key "vol"')

# npy_to_raw.py does the reverse:
#   vol = vol.swapaxes(0, 2)   # → (Nz, Ny, Nx)
#   vol.ravel().tofile(output)
vol_back = vol_npy.swapaxes(0, 2)
raw_out = vol_back.ravel()
print(f'Round-trip identical: {np.array_equal(raw_flat, raw_out)}')

---
## 5. How `compute_transforms.py` builds a camera matrix

For real CT data, each projection is acquired at a known rotation angle θ. The scanner rotates the sample; the X-ray source and detector stay fixed. In the NeRF convention we treat the sample as stationary and rotate the *camera* instead.

The function below is the exact closed-form implementation used in `compute_transforms.py` (function `pose_to_matrix`):

In [ ]:
def pose_to_matrix(theta_deg: float, R: float) -> np.ndarray:
    """Closed-form camera-to-world matrix (matches compute_transforms.py exactly)."""
    th = -math.radians(theta_deg)
    c, s = math.cos(th), math.sin(th)
    R_wc = np.array([[ c, 0,  s],
                     [ s, 0, -c],
                     [ 0, 1,  0]], dtype=float)
    t_w  = np.array([-R * s, R * c, 0.0], dtype=float)
    M = np.eye(4)
    M[:3, :3] = R_wc
    M[:3, 3]  = t_w
    return M

# Verify: look direction is M[:3, 2] (camera +Z in world), should point toward origin
for angle in [0, 45, 90, 180]:
    M = pose_to_matrix(angle, R=5.0)
    origin   = M[:3, 3]
    look_dir = M[:3, 2]   # camera +Z axis in world
    # Camera should look toward the world origin: look_dir ≈ -origin / |origin|
    expected = -origin / np.linalg.norm(origin)
    pointing_toward_origin = np.allclose(look_dir, expected, atol=1e-10)
    print(f"θ={angle:3d}°  origin=({origin[0]:+.2f}, {origin[1]:+.2f}, {origin[2]:+.2f})  "
          f"looks toward origin: {pointing_toward_origin}")

---
## 6. Inspecting real hardware metadata

The `experimental/balls/` directory contains the raw hardware files from a Nikon XT H 225 µCT scanner. Let's look at what `compute_transforms.py` reads.

In [ ]:
# Load .xtekct — scanner geometry and reconstruction parameters
xtekct_path = next((REPO / 'experimental' / 'balls').glob('*.xtekct'))
print(f'File: {xtekct_path.name}\n')

# Matches load_xtekct() in compute_transforms.py
data = {}
current_section = None
for line in xtekct_path.read_text().splitlines():
    if line.startswith('[') and line.endswith(']'):
        current_section = line[1:-1]
        data[current_section] = {}
    elif '=' in line and current_section:
        k, _, v = line.partition('=')
        try:
            v = float(v)
        except ValueError:
            pass
        data[current_section][k] = v

xtek = data.get('XTekCT', {})
src_obj = xtek.get('SrcToObject', float('nan'))
src_det = xtek.get('SrcToDetector', float('nan'))
vox_x   = xtek.get('VoxelSizeX', float('nan'))
vox_n   = xtek.get('VoxelsX', float('nan'))

print(f"Sample name:       {xtek.get('Name', '?')}")
print(f"Source-to-object:  {src_obj} mm")
print(f"Source-to-detector:{src_det} mm")
print(f"Magnification:     {src_det / src_obj:.3f}×")
print(f"Voxel size:        {vox_x} mm")
print(f"Volume:            {int(vox_n)}³ voxels")

# compute_transforms.py computes R automatically from these values:
scale_factor = 2.0 / (vox_x * vox_n)
R_auto = src_obj * scale_factor
print(f"\nAuto-computed camera radius R = {R_auto:.4f} (in NeRF world units)")

In [ ]:
# Load .ang — per-projection rotation angles
# Matches load_from_ang() in compute_transforms.py
ang_path = next((REPO / 'experimental' / 'balls').glob('*.ang'))
print(f'File: {ang_path.name}  ({ang_path.stat().st_size} bytes)\n')

ang_data = {}
lines = ang_path.read_text().splitlines()
for line in lines[1:]:   # skip header
    if not line.strip():
        continue
    idx, angle_str = line.split(':')
    ang_data[int(idx)] = float(angle_str)

angles_arr = np.array(list(ang_data.values()))
print(f'Projections: {len(angles_arr)}')
print(f'Angle range: {angles_arr.min():.2f}° → {angles_arr.max():.2f}°')
print(f'Mean step:   {np.mean(np.diff(angles_arr)):.4f}°')

plt.figure(figsize=(8, 2))
plt.plot(angles_arr, lw=0.8)
plt.xlabel('Projection index'); plt.ylabel('Angle (deg)')
plt.title('Rotation angles from .ang file')
plt.tight_layout()
plt.show()

In [ ]:
# Reconstruct camera positions from .ang angles and compare against the stored transforms.json
exp_transforms_path = REPO / 'experimental' / 'balls' / 'transforms.json'
with open(exp_transforms_path) as f:
    exp_transforms = json.load(f)

# Camera origins from the stored transforms.json (generated by compute_transforms.py)
exp_origins = np.array([fr['transform_matrix'] for fr in exp_transforms['frames']])[:, :3, 3]

# Re-derive positions using pose_to_matrix + the raw .ang angles
# Note: compute_transforms.py applies angle_correction() before calling pose_to_matrix;
# here we use the raw angles so expect a small systematic offset.
recomputed = np.array([pose_to_matrix(a, R_auto)[:3, 3] for a in angles_arr[:len(exp_origins)]])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.scatter(exp_origins[:, 0], exp_origins[:, 1], s=6, label='stored transforms.json', alpha=0.8)
ax.scatter(recomputed[:, 0], recomputed[:, 1], s=6, marker='x', label='re-derived from .ang', alpha=0.6)
ax.scatter(0, 0, s=80, c='red', marker='*')
ax.set_aspect('equal')
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.set_title('Camera ring: stored vs. re-derived (XY plane)')
ax.legend(fontsize=9)

ax2 = axes[1]
pos_diff = np.linalg.norm(exp_origins - recomputed, axis=1)
ax2.plot(pos_diff)
ax2.set_xlabel('Frame index')
ax2.set_ylabel('Position error (world units)')
ax2.set_title('Residual (angle correction in compute_transforms.py\nshifts the ring slightly)')

plt.tight_layout()
plt.show()
print(f'Mean position error: {pos_diff.mean():.4f}  (due to angle_correction(), expected)')

The small residual is the empirical `angle_correction()` applied inside `compute_transforms.py` (a linear correction to account for scanner-specific timing offsets). The ring shape and radius match exactly.

---
## 7. Combining per-timestep transform files

For 4D (time-resolved) datasets, each load step has its own `transforms_NN.json`. `combine_transforms.py` merges them into a single file with a `time` field on every frame.

The script:
1. Globs `transforms_*.json` in the folder
2. Extracts the integer timestamp from the filename stem
3. Applies an optional `--timestamp-func` (e.g. `"lambda x: x/20.0"`)
4. Sets `frame['time']` for every frame in that timestep
5. Drops frames whose image files are missing (prints a warning per dropped frame)
6. Writes `transforms.json`

The snippet below demonstrates the merging logic on a tiny synthetic example.

In [ ]:
import tempfile

base = {
    'camera_angle_x': 0.785, 'fl_x': 500.0, 'fl_y': 500.0,
    'w': 256, 'h': 256, 'cx': 128.0, 'cy': 128.0,
}

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    for step in [0, 10, 20]:
        d = dict(base)
        d['frames'] = [
            {'file_path': f'images_{step:02d}/train_00.png',
             'transform_matrix': np.eye(4).tolist()}
        ]
        (tmpdir / f'transforms_{step:02d}.json').write_text(json.dumps(d))

    # Replicate combine_transforms.py logic (without enforce_exists check)
    timestamp_func = lambda x: x / 20.0
    combined = None
    for fn in sorted(tmpdir.glob('transforms_*.json')):
        timestamp = int(fn.stem.split('_')[-1])
        t = timestamp_func(timestamp)
        d = json.loads(fn.read_text())
        for frame in d['frames']:
            frame['time'] = round(t, 4)
        if combined is None:
            combined = d
        else:
            combined['frames'].extend(d['frames'])

    print('Combined frames:')
    for frame in combined['frames']:
        print(f"  {frame['file_path']:35s}  time={frame['time']}")

---
## Next steps

| Task | Tool |
|---|---|
| Convert `.tiff` projections to `.png` | `scripts/tiff_to_png.py` |
| Generate `transforms.json` from real scan | `scripts/compute_transforms.py` |
| Merge timestep files for 4D training | `scripts/combine_transforms.py` |
| Convert reconstructed volume for NeRF supervision | `scripts/raw_to_npy.py` |
| Inspect a reconstructed volume interactively | `scripts/show_slices.py` |
| Train an X-ray NeRF | [nerfstudio-xray](https://github.com/igrega348/nerfstudio-xray) |

Run any script with `--help` to see all options:
```bash
python scripts/compute_transforms.py --help
```